[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/18_feature_selection/02_wrapper_methods.ipynb)

# Bài 02 — Wrapper methods: forward, backward, RFE

**🎯 Mục tiêu**
- Wrapper: đánh giá **tập con** feature bằng chính model (qua CV) — thấy được tương tác và dư thừa.
- Cài đặt **forward selection** from scratch; dùng `SequentialFeatureSelector`, `RFE`, `RFECV`.
- Chi phí tính toán và rủi ro overfit quá trình chọn; độ ổn định của tập được chọn.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/18_feature_selection

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_friedman1
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, RFECV, SequentialFeatureSelector
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)
X5, y = make_friedman1(n_samples=600, n_features=15, noise=1.0, random_state=1)
X = np.hstack([X5, X5[:, [3]] + rng.normal(0, 0.05, (600, 1))])        # + một bản sao của x4
names = [f"x{i + 1}" for i in range(5)] + [f"noise{i}" for i in range(10)] + ["x4_copy"]
cv = KFold(5, shuffle=True, random_state=0)
model = make_pipeline(SplineTransformer(n_knots=5), RidgeCV())             # model tuyến tính + spline (bắt được x3 hình chữ U)

## 1. Forward selection from scratch

Bắt đầu với tập rỗng; mỗi bước thêm feature làm **điểm CV tăng nhiều nhất**; dừng khi không cải thiện (hoặc đủ $k$).

In [ ]:
def forward_selection(model, X, y, cv, max_features=None, tol=1e-3):
    selected, remaining, history = [], list(range(X.shape[1])), []
    best_score = -np.inf
    while remaining and (max_features is None or len(selected) < max_features):
        scores = {j: cross_val_score(model, X[:, selected + [j]], y, cv=cv, scoring="r2").mean() for j in remaining}
        j_best = max(scores, key=scores.get)
        if scores[j_best] - best_score < tol:
            break
        best_score = scores[j_best]
        selected.append(j_best)
        remaining.remove(j_best)
        history.append((names[j_best], best_score))
    return selected, history


start = time.perf_counter()
selected, history = forward_selection(model, X, y, cv)
print(f"{time.perf_counter() - start:.1f}s")
for i, (f, s) in enumerate(history, 1):
    print(f"bước {i}: thêm {f:8s} → R² CV = {s:.4f}")

Forward selection chọn đúng 5 feature thật, **không** chọn `x4_copy` (đã có x4 — thêm vào không cải thiện) và không chọn nhiễu. Đây là ưu điểm của wrapper so với filter: đánh giá **tập** feature, nên thấy dư thừa.

Chi phí: bước $t$ thử $d - t$ feature × $k$ fold → tổng $O(d^2 k)$ lần train. Với $d$ = 1 000 feature: hàng triệu lần train — không khả thi.

## 2. `SequentialFeatureSelector` (forward / backward)

In [ ]:
for direction in ["forward", "backward"]:
    start = time.perf_counter()
    sfs = SequentialFeatureSelector(model, n_features_to_select="auto", tol=1e-3, direction=direction, cv=cv, scoring="r2").fit(X, y)
    print(f"{direction:8s}: {[n for n, k in zip(names, sfs.get_support()) if k]} | {time.perf_counter() - start:.1f}s")

Backward bắt đầu với **mọi** feature và loại dần — về nguyên tắc tốt hơn khi có tương tác (feature chỉ có ích khi đi cùng feature khác, như $x_1, x_2$), nhưng đắt hơn khi $d$ lớn. Ở đây backward **dừng sớm** và giữ lại nhiều feature nhiễu: bỏ một feature nhiễu chỉ làm điểm CV tăng rất ít (< `tol`), nên tiêu chí dừng kích hoạt. Kết quả của wrapper phụ thuộc mạnh vào tiêu chí dừng — dùng `tol` nhỏ hơn, hoặc cố định `n_features_to_select`, hoặc chọn theo đường cong điểm–số feature như RFECV bên dưới.

## 3. RFE — Recursive Feature Elimination

Train model trên mọi feature → loại feature **kém quan trọng nhất** (theo hệ số / importance của model) → lặp lại. Rẻ hơn SFS (mỗi vòng 1 lần train thay vì $d$ lần), nhưng dựa vào importance của model (có thể lệch).

In [ ]:
rf = RandomForestRegressor(n_estimators=80, random_state=0)
rfecv = RFECV(rf, step=1, cv=cv, scoring="r2", min_features_to_select=1).fit(X, y)
n_feats = np.arange(1, len(rfecv.cv_results_["mean_test_score"]) + 1)
plt.errorbar(n_feats, rfecv.cv_results_["mean_test_score"], yerr=rfecv.cv_results_["std_test_score"], fmt="o-", capsize=3)
plt.axvline(rfecv.n_features_, color="gray", ls="--", label=f"tối ưu: {rfecv.n_features_} feature")
plt.xlabel("số feature")
plt.ylabel("R² (CV)")
plt.legend()
plt.title("RFECV với Random Forest")
plt.show()
print("RFECV giữ:", [n for n, k in zip(names, rfecv.support_) if k])
print("thứ hạng (1 = giữ):", dict(zip(names, rfecv.ranking_)))

RFE dựa vào importance của model nên **giữ cả `x4_copy`** (rừng chia importance giữa x4 và bản sao) — khác với forward selection vốn thấy ngay bản sao là thừa.

Đường cong thường **phẳng** quanh tối ưu: nhiều số lượng feature cho điểm gần như nhau → chọn tập **nhỏ nhất** trong khoảng 1 độ lệch chuẩn của tốt nhất (one-standard-error rule) để có model đơn giản hơn.

## 4. Độ ổn định của tập được chọn

Chạy lại trên các mẫu bootstrap: tập được chọn có giống nhau không?

In [ ]:
counts = pd.Series(0, index=names)
for b in range(10):
    idx = rng.integers(0, len(X), len(X))
    rfe = RFE(RandomForestRegressor(n_estimators=50, random_state=b), n_features_to_select=5, step=2).fit(X[idx], y[idx])
    counts[np.array(names)[rfe.support_]] += 1
(counts / 10).sort_values(ascending=False).head(8).plot.barh()
plt.xlabel("tần suất được chọn (10 bootstrap)")
plt.title("Feature thật được chọn ổn định; x4 và bản sao 'chia nhau' chỗ")
plt.show()

Feature mạnh được chọn gần như mọi lần; với các feature **tương quan** (x4 và `x4_copy`), lựa chọn nhảy qua lại giữa chúng — đừng diễn giải "feature không được chọn = vô dụng".

## 5. Rủi ro: overfit quá trình chọn

Wrapper thử rất nhiều tập con trên cùng dữ liệu → điểm CV của tập tốt nhất bị **lạc quan** (selection bias — chương 15). Muốn ước lượng trung thực: đặt cả quá trình chọn vào **nested CV** (vòng ngoài đánh giá, vòng trong chọn).

In [ ]:
pipe = make_pipeline(SequentialFeatureSelector(model, n_features_to_select=5, direction="forward", cv=3), model)
nested = cross_val_score(pipe, X, y, cv=KFold(4, shuffle=True, random_state=1), scoring="r2")
print(f"nested CV của quy trình 'forward selection 5 feature + model': R² = {nested.mean():.4f} ± {nested.std():.4f}")

## 🧠 Kiểm tra nhanh

1. Wrapper có ưu điểm gì so với filter? Nhược điểm?
2. Forward selection với 50 feature, 5-fold CV, chọn tới 10 feature: khoảng bao nhiêu lần train?
3. Khi nào backward elimination tốt hơn forward?
4. Vì sao nên báo cáo hiệu năng của quy trình chọn feature bằng nested CV?

## 📌 Tóm tắt

- Wrapper chọn tập feature theo điểm CV của chính model → thấy dư thừa, tương tác; đắt $O(d^2)$.
- RFE rẻ hơn, dựa vào importance; RFECV chọn số feature bằng CV; ưu tiên tập nhỏ khi điểm tương đương.
- Tập được chọn không ổn định với feature tương quan; đánh giá quy trình chọn bằng nested CV.

## 📚 Tài liệu tham khảo

- Kohavi & John (1997) — *Wrappers for feature subset selection*
- Guyon et al. (2002) — *Gene Selection for Cancer Classification using Support Vector Machines* (RFE)
- scikit-learn User Guide — Sequential Feature Selection, RFE